# Bài 6 v4 · Cây quyết định
PGS.TS. Nguyễn Văn Hậu. Chạy tuần tự 10 ô mã. Các đồ thị khoa học được tính từ dữ liệu/mô hình; toy và moons được ghi rõ là mô phỏng.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"source/tree_lab.py").exists():raise RuntimeError("Mở notebook tại thư mục bài đã giải nén")
if (ROOT/".build/pydeps").exists():sys.path.insert(0,str(ROOT/".build/pydeps"))
sys.path.insert(0,str(ROOT/"source"))
import json,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier,plot_tree
from tree_lab import toy_data,gini,enumerate_splits,walk,run
from prepare_olist import prepare
from threadpoolctl import threadpool_limits
OUT=ROOT/"results/lab_figures";OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({"figure.figsize":(10,5),"axes.grid":True})

## 1. Đầy đủ dữ liệu và phép chia

In [2]:
X,y=toy_data();print(pd.DataFrame({"ID":[f"M{i:02}" for i in range(1,13)],"x_100km":X[:,0],"h_days":X[:,1],"y":y}).to_string(index=False))
print("Gini gốc",gini(y));candidates=enumerate_splits(X,y);print(pd.DataFrame(candidates).to_string(index=False))
fig,ax=plt.subplots();ax.scatter(X[:,0],X[:,1],c=np.where(y,"#bc761a","#245a84"));ax.axhline(15,color="#167a72");ax.set(xlabel="x (100 km), synthetic",ylabel="promised days");fig.tight_layout();fig.savefig(OUT/"toy_root.png",dpi=240);plt.show()

 ID  x_100km  h_days  y
M01      1.0    10.0  1
M02      2.0    20.0  0
M03      3.0    20.0  0
M04      4.0    10.0  1
M05      5.0    20.0  0
M06      6.0    20.0  0
M07      7.0    10.0  1
M08      8.0    20.0  0
M09      9.0    20.0  0
M10     10.0    10.0  1
M11     11.0    20.0  1
M12     12.0    20.0  1
Gini gốc 0.5
 feature  threshold  n_left  n_right  pos_left  pos_right   g_left  g_right    after     gain
       0        1.5       1       11         1          5 0.000000 0.495868 0.454545 0.045455
       0        2.5       2       10         1          5 0.500000 0.500000 0.500000 0.000000
       0        3.5       3        9         1          5 0.444444 0.493827 0.481481 0.018519
       0        4.5       4        8         2          4 0.500000 0.500000 0.500000 0.000000
       0        5.5       5        7         2          4 0.480000 0.489796 0.485714 0.014286
       0        6.5       6        6         2          4 0.444444 0.444444 0.444444 0.055556
       0        7

C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\1094050742.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();ax.scatter(X[:,0],X[:,1],c=np.where(y,"#bc761a","#245a84"));ax.axhline(15,color="#167a72");ax.set(xlabel="x (100 km), synthetic",ylabel="promised days");fig.tight_layout();fig.savefig(OUT/"toy_root.png",dpi=240);plt.show()


In [3]:
best=max(candidates,key=lambda c:c["gain"]);right=X[:,1]>15;print("Best root",best);print("Right subset",enumerate_splits(X[right],y[right]))
toy=DecisionTreeClassifier(random_state=42).fit(X,y);fig,ax=plt.subplots(figsize=(12,6));plot_tree(toy,feature_names=["x_100km","h_days"],class_names=["on time","late"],filled=True,ax=ax);fig.tight_layout();fig.savefig(OUT/"toy_tree.png",dpi=240);plt.show()
assert toy.get_n_leaves()==3 and toy.get_depth()==2

Best root {'feature': 1, 'threshold': 15.0, 'n_left': 4, 'n_right': 8, 'pos_left': 4, 'pos_right': 2, 'g_left': 0.0, 'g_right': 0.375, 'after': 0.25, 'gain': 0.25}
Right subset [{'feature': 0, 'threshold': 2.5, 'n_left': 1, 'n_right': 7, 'pos_left': 0, 'pos_right': 2, 'g_left': 0.0, 'g_right': 0.40816326530612246, 'after': 0.35714285714285715, 'gain': 0.01785714285714285}, {'feature': 0, 'threshold': 4.0, 'n_left': 2, 'n_right': 6, 'pos_left': 0, 'pos_right': 2, 'g_left': 0.0, 'g_right': 0.4444444444444444, 'after': 0.3333333333333333, 'gain': 0.041666666666666685}, {'feature': 0, 'threshold': 5.5, 'n_left': 3, 'n_right': 5, 'pos_left': 0, 'pos_right': 2, 'g_left': 0.0, 'g_right': 0.48, 'after': 0.3, 'gain': 0.07500000000000001}, {'feature': 0, 'threshold': 7.0, 'n_left': 4, 'n_right': 4, 'pos_left': 0, 'pos_right': 2, 'g_left': 0.0, 'g_right': 0.5, 'after': 0.25, 'gain': 0.125}, {'feature': 0, 'threshold': 8.5, 'n_left': 5, 'n_right': 3, 'pos_left': 0, 'pos_right': 2, 'g_left': 0.0, '

C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\1606607011.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  toy=DecisionTreeClassifier(random_state=42).fit(X,y);fig,ax=plt.subplots(figsize=(12,6));plot_tree(toy,feature_names=["x_100km","h_days"],class_names=["on time","late"],filled=True,ax=ax);fig.tight_layout();fig.savefig(OUT/"toy_tree.png",dpi=240);plt.show()


## 2. Cắt tỉa thực và trường hợp hòa

In [4]:
path=toy.cost_complexity_pruning_path(X,y);print("alpha",path.ccp_alphas,"R",path.impurities)
for alpha in [.1,.25,.3]:print(alpha,[3*alpha,.25+2*alpha,.5+alpha])
fig,ax=plt.subplots();alphas=np.linspace(0,.4,100)
for L,r in [(3,0),(2,.25),(1,.5)]:ax.plot(alphas,r+alphas*L,label=f"{L} leaves")
ax.set(xlabel="alpha",ylabel="R + alpha*L");ax.legend();fig.tight_layout();fig.savefig(OUT/"pruning_objective.png",dpi=240);plt.show()

alpha [0.   0.25] R [0.  0.5]
0.1 [0.30000000000000004, 0.45, 0.6]
0.25 [0.75, 0.75, 0.75]
0.3 [0.8999999999999999, 0.85, 0.8]


C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\3183586241.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel="alpha",ylabel="R + alpha*L");ax.legend();fig.tight_layout();fig.savefig(OUT/"pruning_objective.png",dpi=240);plt.show()


## 3. Dữ liệu thật và protocol khóa trước báo cáo test

In [5]:
provenance=prepare()
with threadpool_limits(limits=1):audit=run()
print("Data",provenance["split_counts"],"unknown weights",provenance["weight_correction_count"])
print("Policy",audit["policy"]);print("Candidates",[(c["depth"],c["validation"]["ap"]) for c in audit["candidates"]])
assert audit["status"]=="pass"

{"status": "pass", "raw_orders": 99441, "eligible_completed_orders": 96394, "split_counts": {"test": 40266, "train": 29031, "validation": 10389}, "positive_counts": {"test": 3478, "train": 941, "validation": 634}, "weight_correction_count": 16, "weight_corrections_by_split": {"train": 16}, "missing_by_split": {"test": 0, "train": 16, "validation": 0}, "other_columns_preserved": true, "weight_rule": "Total weight is unknown if any item weight is unknown; do not replace unknown with zero.", "distance_recipe": "Median coordinate per postal prefix; haversine radius6371.0088km; mean item distance per order.", "label": "date(delivered)>date(estimated); same calendar day is on time.", "sha256_raw": "967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784", "sha256_prepared": "bdabb4fc0f6c36102aebb0691cc539a7ed6b698084917ec330429bd63ce1d4b1", "limitations": ["Completed-only eligible cohort; train/validation labels complete before cutoff.", "Static product/geographic metadata: historic

{"status": "pass", "checks": 32, "tree_depth": 6, "threshold": 0.1, "test": {"n": 40266, "tn": 34650, "fp": 2138, "fn": 3046, "tp": 432, "alerts": 2570, "precision": 0.1680933852140078, "precision_defined": true, "recall": 0.1242093156986774, "accuracy": 0.8712561466249441, "f1": 0.14285714285714285, "cost": 17368, "ap": 0.1339855500419443, "auc": 0.6158664501569161, "brier": 0.0805576711296006, "log_loss": 0.3197830468800299}, "paired": {"both_correct": 34713, "tree_fixes": 369, "tree_new_errors": 1877, "both_wrong": 3307, "fn_fixed": 299, "fn_new": 28, "fp_fixed": 70, "fp_new": 1849}}
Data {'test': 40266, 'train': 29031, 'validation': 10389} unknown weights 16
Policy {'features': ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g'], 'criterion': 'gini', 'depth_candidates': [3, 6], 'selected_depth': 6, 'min_samples_leaf': 200, 'ccp_alpha': 0, 'random_state': 42, 'class_weight': None, 'preprocessing': 'Median fit train only; original units; no scaler', 'str

In [6]:
scan=pd.DataFrame(audit["validation_scan"]);print(scan[["threshold","fp","fn","alerts","cost"]].to_string(index=False))
fig,ax=plt.subplots();points=scan[scan.threshold.notna()];ax.plot(points.threshold,points.cost,marker="o");ax.axhline(scan.loc[scan.threshold.isna(),"cost"].iloc[0],ls="--",color="gray",label="no alerts");ax.set(xlabel="tau (validation)",ylabel="hypothetical K=FP+5FN");ax.legend();fig.tight_layout();fig.savefig(OUT/"validation_policy.png",dpi=240);plt.show()

 threshold   fp  fn  alerts  cost
      0.02 9247  11    9870  9302
      0.03 2090 420    2304  4190
      0.05  850 509     975  3395
      0.10  287 569     352  3132
      0.15    0 634       0  3170
      0.20    0 634       0  3170
      0.30    0 634       0  3170
      0.50    0 634       0  3170
      0.80    0 634       0  3170
       NaN    0 634       0  3170


C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\2168704948.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();points=scan[scan.threshold.notna()];ax.plot(points.threshold,points.cost,marker="o");ax.axhline(scan.loc[scan.threshold.isna(),"cost"].iloc[0],ls="--",color="gray",label="no alerts");ax.set(xlabel="tau (validation)",ylabel="hypothetical K=FP+5FN");ax.legend();fig.tight_layout();fig.savefig(OUT/"validation_policy.png",dpi=240);plt.show()


In [7]:
o=audit["one_order"];nodes=json.loads((ROOT/"results/chosen_tree.json").read_text());print("ID",o["order_id"]);print(pd.DataFrame(o["path"]).to_string(index=False));print("Leaf",o["leaf_counts"]);print("Policy",audit["policy"]["selected_threshold"],"truth",o["late"])
leaf,p,path=walk(nodes,o["filled"]);assert leaf==o["leaf"] and np.isclose(p,o["p"])
print("Test",audit["test_selected"],"Logistic",audit["logistic"]["test_selected"])

ID fc9bc4bac939adaa600ad59dee9a106f
 node  feature      value   threshold  go_left
    0        0 100.028000 1395.273621     True
    1        1   7.197801    9.544618     True
Leaf {'node': 2, 'left': -1, 'right': -1, 'feature': -2, 'threshold': None, 'n': 297, 'positive': 33, 'p': 0.1111111111111111, 'gini': 0.19753086419753085}
Policy 0.1 truth 0
Test {'n': 40266, 'tn': 34650, 'fp': 2138, 'fn': 3046, 'tp': 432, 'alerts': 2570, 'precision': 0.1680933852140078, 'precision_defined': True, 'recall': 0.1242093156986774, 'accuracy': 0.8712561466249441, 'f1': 0.14285714285714285, 'cost': 17368, 'ap': 0.1339855500419443, 'auc': 0.6158664501569161, 'brier': 0.0805576711296006, 'log_loss': 0.3197830468800299} Logistic {'n': 40266, 'tn': 36429, 'fp': 359, 'fn': 3317, 'tp': 161, 'alerts': 520, 'precision': 0.3096153846153846, 'precision_defined': True, 'recall': 0.04629097182288672, 'accuracy': 0.9087070977996324, 'f1': 0.08054027013506754, 'cost': 16944, 'ap': 0.15923670168606474, 'auc': 0.656

## 4. Đọc lỗi hai chiều và giải thích mô hình

In [8]:
print("Paired error counts",audit["paired_counts"]);print("Deterministic examples",audit["paired_examples"])
fig,ax=plt.subplots();labels=["FN fixed","FN new","FP fixed","FP new"];keys=["fn_fixed","fn_new","fp_fixed","fp_new"];ax.bar(labels,[audit["paired_counts"][k] for k in keys],color=["#167a72","#bc761a","#167a72","#bc761a"]);ax.set(ylabel="orders, full historical test");fig.tight_layout();fig.savefig(OUT/"paired_errors.png",dpi=240);plt.show()

Paired error counts {'both_correct': 34713, 'tree_fixes': 369, 'tree_new_errors': 1877, 'both_wrong': 3307, 'fn_fixed': 299, 'fn_new': 28, 'fp_fixed': 70, 'fp_new': 1849}
Deterministic examples {'both_correct': [{'order_id': '000229ec398224ef6ca0657da4fc703e', 'late': 0, 'tree_p': 0.027678985619297974, 'tree_prediction': 0, 'logistic_p': 0.02830979937047545, 'logistic_prediction': 0}, {'order_id': '00061f2a7bc09da83e415a52dc8a4af1', 'late': 0, 'tree_p': 0.03076923076923077, 'tree_prediction': 0, 'logistic_p': 0.033721829144680866, 'logistic_prediction': 0}], 'tree_fixes': [{'order_id': '00254baeb6c932b0a8aeead91fbd02b5', 'late': 0, 'tree_p': 0.08360655737704918, 'tree_prediction': 0, 'logistic_p': 0.10109803209059225, 'logistic_prediction': 1}, {'order_id': '0084e195fbd72ae51599af47f04afede', 'late': 1, 'tree_p': 0.13333333333333333, 'tree_prediction': 1, 'logistic_p': 0.05530950771120421, 'logistic_prediction': 0}], 'tree_new_errors': [{'order_id': '0005a1a1728c9d785b8e2b08b904576c', 

C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\1290734334.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots();labels=["FN fixed","FN new","FP fixed","FP new"];keys=["fn_fixed","fn_new","fp_fixed","fp_new"];ax.bar(labels,[audit["paired_counts"][k] for k in keys],color=["#167a72","#bc761a","#167a72","#bc761a"]);ax.set(ylabel="orders, full historical test");fig.tight_layout();fig.savefig(OUT/"paired_errors.png",dpi=240);plt.show()


In [9]:
fig,axs=plt.subplots(1,2,figsize=(13,5));features=audit["features"];axs[0].barh(features,audit["mdi"]);axs[0].set(xlabel="MDI (train)");axs[1].barh(features,audit["permutation_mean"],xerr=audit["permutation_std"]);axs[1].set(xlabel="AP drop (validation), mean +/- std of 5 permutations");fig.tight_layout();fig.savefig(OUT/"importance.png",dpi=240);plt.show()
print("Feature importance and decision rules do not establish causality.")

Feature importance and decision rules do not establish causality.


C:\Users\nguye\AppData\Local\Temp\ipykernel_1452\1345615369.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,axs=plt.subplots(1,2,figsize=(13,5));features=audit["features"];axs[0].barh(features,audit["mdi"]);axs[0].set(xlabel="MDI (train)");axs[1].barh(features,audit["permutation_mean"],xerr=audit["permutation_std"]);axs[1].set(xlabel="AP drop (validation), mean +/- std of 5 permutations");fig.tight_layout();fig.savefig(OUT/"importance.png",dpi=240);plt.show()


In [10]:
from science_audit import run as independent_audit
report=independent_audit();assert report["status"]=="pass"
print("Independent checks",report["checks_count"])
print("Exit: G_right=3/8; gain at node=3/8; global contribution=(8/12)*(3/8)=1/4.")
print("Conclusion: tree trades higher recall for more false alerts on this historical cohort. Choose configuration on validation; keep test history disclosed.")

{"status": "pass", "checks": 169}
Independent checks 169
Exit: G_right=3/8; gain at node=3/8; global contribution=(8/12)*(3/8)=1/4.
Conclusion: tree trades higher recall for more false alerts on this historical cohort. Choose configuration on validation; keep test history disclosed.
